In [ ]:
import pandas as pd
import numpy as np

# Load data from CSV
df = pd.read_csv('house-price-data.csv')

# Show first 5 rows
df.head()

In [ ]:
import matplotlib.pyplot as plt

# 3D scatter plot - visualize the data
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

ax.scatter(df['area'], df['bedrooms'], df['price'], alpha=0.5)

ax.set_xlabel('Area (1000 sqft)')
ax.set_ylabel('Bedrooms')
ax.set_zlabel('Price (lakhs)')
ax.set_title('House Price Data')

plt.show()

In [ ]:
from ipywidgets import interact, FloatSlider

area_range = np.linspace(df['area'].min(), df['area'].max(), 20)
bedroom_range = np.linspace(df['bedrooms'].min(), df['bedrooms'].max(), 20)
area_grid, bedroom_grid = np.meshgrid(area_range, bedroom_range)

def plot_plane(w1, w2, bias):
    fig = plt.figure(figsize=(10, 7))
    ax = fig.add_subplot(111, projection='3d')

    ax.scatter(df['area'], df['bedrooms'], df['price'], alpha=0.5)

    price_grid = w1 * area_grid + w2 * bedroom_grid + bias
    ax.plot_surface(area_grid, bedroom_grid, price_grid, alpha=0.3, color='red')

    ax.set_xlabel('Area (1000 sqft)')
    ax.set_ylabel('Bedrooms')
    ax.set_zlabel('Price (lakhs)')
    ax.set_title(f'price = {w1:.1f} * area + {w2:.1f} * bedrooms + {bias:.1f}')
    ax.set_zlim(df['price'].min() - 5, df['price'].max() + 5)

    plt.show()

interact(plot_plane,
         w1=FloatSlider(value=0, min=-5, max=20, step=0.5, description='w1 (area)'),
         w2=FloatSlider(value=0, min=-5, max=15, step=0.5, description='w2 (beds)'),
         bias=FloatSlider(value=0, min=-10, max=30, step=0.5, description='bias'))

In [ ]:
# Prepare features and target
X = df[['area', 'bedrooms']].values
y = df['price'].values

In [ ]:
class LinearRegression:
    def __init__(self, learning_rate=0.01, n_epochs=1000):
        self.learning_rate = learning_rate
        self.n_epochs = n_epochs
        self.weights = None
        self.bias = None

    def fit(self, X, y):
        # Initialize parameters
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0

        for _ in range(self.n_epochs):

            # Predict
            y_predicted = np.dot(X, self.weights) + self.bias

            # Calculate gradients
            dw = (1 / n_samples) * np.dot(X.T, (y_predicted - y))
            db = (1 / n_samples) * np.sum(y_predicted - y)

            # Update parameters
            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db

            # Calculate loss
            loss = np.mean((y_predicted - y) ** 2)

    def predict(self, X):
        return np.dot(X, self.weights) + self.bias

In [ ]:
# Create and train model
model = LinearRegression(learning_rate=0.01, n_epochs=1000)
model.fit(X, y)

# Make predictions
predictions = model.predict([[1.0, 2], [1.5, 3], [2.0, 4], [2.5, 5]])

# Print results
print("Predictions:", predictions)
print("Learned weights:", model.weights)
print("Learned bias:", model.bias)

In [ ]:
# 3D plot with the fitted plane
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

# Scatter plot of actual data
ax.scatter(df['area'], df['bedrooms'], df['price'], alpha=0.5, label='Data points')

# Create a grid for the plane
area_range = np.linspace(df['area'].min(), df['area'].max(), 20)
bedroom_range = np.linspace(df['bedrooms'].min(), df['bedrooms'].max(), 20)
area_grid, bedroom_grid = np.meshgrid(area_range, bedroom_range)

# Calculate price on the plane: price = w1 * area + w2 * bedrooms + bias
price_grid = model.weights[0] * area_grid + model.weights[1] * bedroom_grid + model.bias

# Draw the fitted plane
ax.plot_surface(area_grid, bedroom_grid, price_grid, alpha=0.3, color='red')

ax.set_xlabel('Area (1000 sqft)')
ax.set_ylabel('Bedrooms')
ax.set_zlabel('Price (lakhs)')
ax.set_title('Fitted Plane: price = w1 * area + w2 * bedrooms + bias')

plt.show()